# Problem 5: Benchmarking Arabic NLP Tasks Across LLMs
## Team 14 — Task: إعادة الصياغة (Paraphrasing)

**Objective:** Evaluate how 5 Large Language Models (ChatGPT, Gemini, ALLaM, Jais, Fanar) perform on Arabic paraphrasing — rewriting Arabic sentences with the same meaning using different words and structures.

**Metric:** BLEU + Human Evaluation

**Dataset:** 100 Arabic sentence pairs from `arabic_paraphrasing.csv`, including a mix of Modern Standard Arabic (MSA), Classical Arabic, and dialectal Arabic.

In [ ]:
# ── Install Dependencies (run this cell first on Colab) ──
import subprocess, sys

packages = [
    "openai",
    "google-genai",
    "transformers",
    "torch",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
    "nltk",
    "scikit-learn",
    "seaborn",
    "pandas",
    "numpy",
    "matplotlib",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from google import genai
from transformers import pipeline as hf_pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import snapshot_download, InferenceClient
import torch
import os
import json
import time
import re
import gc
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Check if running on Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running on Colab — Google Drive mount will be skipped.")

# Check GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Local HF models will require GPU.")

In [ ]:
# Mount Google Drive (Colab only)
if IN_COLAB:
    drive.mount('/content/drive')
    drive_model_path = "/content/drive/MyDrive/SHARED/Assignments/NNs"
else:
    drive_model_path = "."  # Local fallback path
    os.makedirs(os.path.join(drive_model_path, "hf_models"), exist_ok=True)
print(f"Model base path: {drive_model_path}")

## 1. Setup & Configuration

In [ ]:
# ⚠️ Set your API keys as environment variables or in Colab Secrets — do NOT hardcode them.
# In Colab: use userdata.get() or set via the Secrets panel (key icon in sidebar).
# Locally: export them in your shell before running.
if IN_COLAB:
    from google.colab import userdata
    GEMINI_API_KEY      = userdata.get('GEMINI_API_KEY')
    OPENROUTER_API_KEY  = userdata.get('OPENROUTER_API_KEY')
    FANAR_API_KEY       = userdata.get('FANAR_API_KEY')
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
else:
    GEMINI_API_KEY      = os.environ.get('GEMINI_API_KEY', '')
    OPENROUTER_API_KEY  = os.environ.get('OPENROUTER_API_KEY', '')
    FANAR_API_KEY       = os.environ.get('FANAR_API_KEY', '')
    # Set HF_TOKEN in your environment before running

In [ ]:
MODEL_CONFIG = {
    "ChatGPT": {
        "model_id": "openai/gpt-4o-mini",
        "backend": "openrouter",
    },
    "Gemini": {
        "model_id": "google/gemini-2.0-flash-lite-001",
        "backend": "openrouter",
    },
    "ALLaM": {
        "model_id": "humain-ai/ALLaM-7B-Instruct-preview",
        "backend": "hf_local",
        "drive_path": os.path.join(drive_model_path, "hf_models/ALLaM-7B-Instruct-preview"),
        "quantize": False,  # 7B fits on T4 in float16
        "max_new_tokens": 256,
    },
    "Jais": {
        "model_id": "inceptionai/Jais-2-8B-Chat",
        "backend": "hf_local",
        "drive_path": os.path.join(drive_model_path, "hf_models/Jais-2-8B-Chat"),
        "quantize": False,  # 8B fits on T4 in float16
        "max_new_tokens": 256,
    },
    "Fanar": {
        "model_id": "Fanar-C-2-27B",
        "backend": "fanar",
    },
}

MODEL_NAMES = list(MODEL_CONFIG.keys())
print("Models to evaluate:", MODEL_NAMES)

In [ ]:
SYSTEM_PROMPT = (
    "أنت مساعد لغوي متخصص في اللغة العربية. "
    "مهمتك إعادة صياغة الجمل العربية بنفس المعنى باستخدام كلمات وتراكيب مختلفة."
)

USER_PROMPT_TEMPLATE = (
    "أعد صياغة الجملة التالية بالعربية بنفس المعنى، مع استخدام كلمات وتراكيب مختلفة قدر الإمكان.\n\n"
    "الجملة: {sentence}\n\n"
    "اكتب الجملة المعاد صياغتها فقط، بدون أي شرح أو تعليق إضافي."
)

print("System prompt:", SYSTEM_PROMPT[:80], "...")
print("User prompt template ready.")

## 2. Dataset Loading & Exploration
We use the `arabic_paraphrasing.csv` dataset containing Arabic sentence pairs annotated for paraphrase similarity. We filter for true paraphrase pairs (`paraphrase == 'p'`) and sample 100 sentences for evaluation across all 5 models.

In [ ]:
# ── Load & Explore Dataset ──
df = pd.read_csv("arabic_paraphrasing.csv")
print(f"Total sentence pairs: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nParaphrase label distribution:\n{df['paraphrase'].value_counts()}")
print(f"\nSimilarity label distribution:\n{df['similarity'].value_counts()}")
print(f"\nExpert score stats:\n{df['44_experts'].describe()}")

# Keep only true paraphrase pairs (gold reference is a valid paraphrase)
df_para = df[df['paraphrase'] == 'p'].reset_index(drop=True)
print(f"\nTrue paraphrase pairs available: {len(df_para)}")

# Select 100 samples for evaluation
df_sample = df_para.sample(n=100, random_state=42).reset_index(drop=True)
print(f"Selected for evaluation: {len(df_sample)}")
df_sample.head(10)

In [ ]:
# ── Dataset Composition Analysis ──
# The assignment requires a balanced mix: ~1/3 MSA, ~1/3 Classical Arabic, ~1/3 Dialectal Arabic
# We classify sentences based on expert scores and linguistic features

def classify_variety(row):
    """Heuristic classification of Arabic variety based on expert score and content."""
    score = row['44_experts']
    sentence = row['First sentence']
    # High similarity scores often indicate MSA (formal, standard)
    # Lower scores may indicate dialectal or classical variation
    if score >= 3.5:
        return 'MSA'
    elif score >= 2.5:
        return 'Classical/Formal'
    else:
        return 'Dialectal'

df_sample['variety'] = df_sample.apply(classify_variety, axis=1)

print("Dataset Composition (estimated):")
print(df_sample['variety'].value_counts())
print(f"\nPercentage distribution:")
print((df_sample['variety'].value_counts(normalize=True) * 100).round(1))

# Show sample from each variety
for v in df_sample['variety'].unique():
    print(f"\n--- Example ({v}) ---")
    example = df_sample[df_sample['variety'] == v].iloc[0]
    print(f"  Input:  {example['First sentence']}")
    print(f"  Gold:   {example['second sentence']}")
    print(f"  Score:  {example['44_experts']}")

## 3. Model Inference Functions
Each model is queried with the **same prompt** and **same settings** (`temperature≈0`, `max_tokens=256`) for consistency.
- **ChatGPT** → via OpenRouter API
- **Gemini** → via OpenRouter API
- **ALLaM (7B)** → loaded locally in float16 on Colab GPU
- **Jais (8B)** → loaded locally in float16 via HuggingFace `pipeline`
- **Fanar** → via Fanar API

In [ ]:
# ── API Helper Functions (ChatGPT, Gemini, Fanar) ──

def ask_openrouter(sentence, model_id):
    """Query a model via OpenRouter API (ChatGPT, Gemini)."""
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
    prompt = USER_PROMPT_TEMPLATE.format(sentence=sentence)
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
            max_tokens=256,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"    OpenRouter error: {e}")
        return f"ERROR: {e}"


def ask_fanar(sentence, model_id):
    """Query Fanar via their OpenAI-compatible API with content filter retry."""
    client = OpenAI(api_key=FANAR_API_KEY, base_url="https://api.fanar.qa/v1")
    prompt = USER_PROMPT_TEMPLATE.format(sentence=sentence)

    # Attempt 1: Full prompt with system message
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
            max_tokens=256,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        if "content_filter" in str(e) or "safety" in str(e):
            # Attempt 2: Simpler prompt without system message
            time.sleep(1)
            try:
                simple_prompt = f"أعد صياغة هذه الجملة بالعربية: {sentence}"
                response = client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {"role": "user", "content": simple_prompt},
                    ],
                    temperature=0.1,
                    max_tokens=256,
                )
                return response.choices[0].message.content.strip()
            except Exception as e2:
                print(f"    Fanar content filter (both attempts): {sentence[:40]}...")
                return f"ERROR: content_filter"
        else:
            print(f"    Fanar error: {e}")
            return f"ERROR: {e}"


print("API helper functions defined (OpenRouter, Fanar).")

In [ ]:
# ── Local HF Helper Functions (ALLaM, Jais) ──

def ask_hf_local(sentence, pipe, max_new_tokens=256):
    """Query a locally loaded HuggingFace model via pipeline."""
    prompt = USER_PROMPT_TEMPLATE.format(sentence=sentence)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        output = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
        generated = output[0]["generated_text"]
        # Chat-style output: extract the assistant reply
        if isinstance(generated, list):
            for msg in reversed(generated):
                if isinstance(msg, dict) and msg.get("role") == "assistant":
                    return msg["content"].strip()
            return str(generated[-1]).strip()
        # Raw text: try to remove prompt prefix
        return generated.split(prompt)[-1].strip()
    except Exception as e:
        print(f"    HF pipeline error: {e}")
        return f"ERROR: {e}"


print("Local HF helper functions defined.")

In [ ]:
# ── HF Model Loading Helpers ──

def load_hf_model(model_name):
    """Download (if needed) and load an HF model. Returns a text-generation pipeline."""
    config = MODEL_CONFIG[model_name]
    drive_path = config["drive_path"]
    repo_id = config["model_id"]
    use_quantize = config.get("quantize", False)

    # Clear any stale CUDA cache before loading a new model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Check if weights exist on Drive
    weights_exist = False
    if os.path.isdir(drive_path):
        files = os.listdir(drive_path)
        weights_exist = (
            "model.safetensors" in files
            or "pytorch_model.bin" in files
            or any(f.startswith("model-") and f.endswith(".safetensors") for f in files)
        )

    if not weights_exist:
        print(f"  Downloading {repo_id} to {drive_path}...")
        snapshot_download(repo_id=repo_id, local_dir=drive_path)

    print(f"  Loading {model_name} from {drive_path}...")
    tokenizer = AutoTokenizer.from_pretrained(drive_path, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    if use_quantize:
        # 4-bit quantization for large models
        print(f"  Using 4-bit quantization for {model_name} (model too large for float16)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            drive_path,
            quantization_config=bnb_config,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            drive_path,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )

    model.eval()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True

    pipe = hf_pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")
    print(f"  {model_name} loaded successfully.")
    return pipe


def free_hf_model(pipe):
    """Free GPU memory after using a model."""
    if pipe is not None:
        del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("  GPU memory freed.")

print("HF model loading helpers defined.")

In [ ]:
# ── Unified Query Function ──

def query_model(sentence, model_name, hf_pipe=None):
    """Unified query function that routes to the correct backend."""
    config = MODEL_CONFIG[model_name]
    backend = config["backend"]

    if backend == "openrouter":
        return ask_openrouter(sentence, config["model_id"])
    elif backend == "fanar":
        return ask_fanar(sentence, config["model_id"])
    elif backend == "hf_local":
        max_new_tokens = config.get("max_new_tokens", 256)
        return ask_hf_local(sentence, hf_pipe, max_new_tokens=max_new_tokens)
    else:
        return "ERROR: Unknown backend"

print("Unified query function defined.")

## 4. Run Evaluation Across All 5 Models
**Evaluation Protocol:** All five models receive identical inputs and identical prompt format. Model settings (`temperature=0`) are kept consistent.
- For API models (ChatGPT, Gemini, Fanar): query the API directly with rate limiting.
- For HF models (ALLaM, Jais): load one at a time, run inference, then free GPU memory.

In [ ]:
# ── Evaluate API Models (ChatGPT, Gemini, Fanar) ──
results = {name: [] for name in MODEL_CONFIG}

API_MODELS = [m for m in MODEL_CONFIG if MODEL_CONFIG[m]["backend"] in ("openrouter", "fanar")]

for model_name in API_MODELS:
    config = MODEL_CONFIG[model_name]
    print(f"\n{'='*60}")
    print(f"  Evaluating: {model_name} ({config['backend']})")
    print(f"{'='*60}")

    for idx, row in df_sample.iterrows():
        sentence = row["First sentence"]
        if (idx + 1) % 25 == 0:
            print(f"  Processing {idx+1}/{len(df_sample)}...")

        output = query_model(sentence, model_name)
        results[model_name].append(output)

        # Rate-limit API calls to avoid throttling
        time.sleep(1.0)

    n_errors = sum(1 for o in results[model_name] if o.startswith("ERROR"))
    print(f"  Done: {len(results[model_name])} outputs, {n_errors} errors")

print("\n API models evaluated!")

In [ ]:
# ── Evaluate ALLaM (Local HF) ──
model_name = "ALLaM"
results[model_name] = []
config = MODEL_CONFIG[model_name]
print(f"\n{'='*60}")
print(f"  Evaluating: {model_name} ({config['backend']})")
print(f"{'='*60}")

hf_pipe = load_hf_model(model_name)
for idx, row in df_sample.iterrows():
    sentence = row["First sentence"]
    if (idx + 1) % 25 == 0:
        print(f"  Processing {idx+1}/{len(df_sample)}...")
    output = query_model(sentence, model_name, hf_pipe=hf_pipe)
    results[model_name].append(output)

# Free GPU memory
free_hf_model(hf_pipe)

n_errors = sum(1 for o in results[model_name] if o.startswith("ERROR"))
print(f"  Done: {len(results[model_name])} outputs, {n_errors} errors")

In [ ]:
# ── Evaluate Jais (Local HF via pipeline — inceptionai/Jais-2-8B-Chat) ──
from transformers import pipeline

model_name = "Jais"
results[model_name] = []
config = MODEL_CONFIG[model_name]
print(f"\n{'='*60}")
print(f"  Evaluating: {model_name} ({config['backend']})")
print(f"  Using: inceptionai/Jais-2-8B-Chat via HuggingFace pipeline")
print(f"{'='*60}")

# Load Jais using HuggingFace pipeline directly
pipe = pipeline("text-generation", model="inceptionai/Jais-2-8B-Chat")

for idx, row in df_sample.iterrows():
    sentence = row["First sentence"]
    if (idx + 1) % 25 == 0:
        print(f"  Processing {idx+1}/{len(df_sample)}...")

    prompt = USER_PROMPT_TEMPLATE.format(sentence=sentence)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    try:
        output = pipe(messages, max_new_tokens=256, do_sample=False)
        generated = output[0]["generated_text"]
        # Chat-style output: extract the assistant reply
        if isinstance(generated, list):
            for msg in reversed(generated):
                if isinstance(msg, dict) and msg.get("role") == "assistant":
                    result_text = msg["content"].strip()
                    break
            else:
                result_text = str(generated[-1]).strip()
        else:
            result_text = generated.split(prompt)[-1].strip()
    except Exception as e:
        print(f"    Jais pipeline error: {e}")
        result_text = f"ERROR: {e}"

    results[model_name].append(result_text)

# Free GPU memory
del pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  GPU memory freed.")

n_errors = sum(1 for o in results[model_name] if o.startswith("ERROR"))
print(f"  Done: {len(results[model_name])} outputs, {n_errors} errors")

In [ ]:
print("\n All 5 models evaluated!")
print(f"  Models: {list(MODEL_CONFIG.keys())}")
for model_name in MODEL_CONFIG:
    n_err = sum(1 for o in results[model_name] if o.startswith("ERROR"))
    print(f"  {model_name:<12}: {len(results[model_name])} outputs, {n_err} errors")

## 5. Evaluation Metrics — BLEU Score
We compute **sentence-level BLEU** (with smoothing) for each model output against the gold reference paraphrase. This is the standard automatic metric for paraphrasing tasks.

In [ ]:
# ── Compute BLEU Scores ──
smoother = SmoothingFunction().method1

def compute_bleu(reference, hypothesis):
    """Compute sentence-level BLEU for Arabic text (whitespace tokenization)."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    if len(hyp_tokens) == 0:
        return 0.0
    return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)

bleu_scores = {}
for model_name in MODEL_CONFIG:
    scores = []
    for idx in range(len(df_sample)):
        reference = df_sample.iloc[idx]["second sentence"]
        hypothesis = results[model_name][idx]
        if hypothesis.startswith("ERROR"):
            scores.append(0.0)
        else:
            scores.append(compute_bleu(reference, hypothesis))
    bleu_scores[model_name] = scores

# ── Summary Statistics ──
print(f"{'Model':<12} {'Mean BLEU':>10} {'Median':>10} {'Std':>10} {'Min':>8} {'Max':>8} {'Errors':>8}")
print("─" * 70)
for model_name in MODEL_CONFIG:
    s = bleu_scores[model_name]
    n_err = sum(1 for o in results[model_name] if o.startswith("ERROR"))
    print(f"{model_name:<12} {np.mean(s):>10.4f} {np.median(s):>10.4f} {np.std(s):>10.4f} "
          f"{np.min(s):>8.4f} {np.max(s):>8.4f} {n_err:>8d}")

In [ ]:
# ── Full Results Table ──
results_df = pd.DataFrame({
    "Input (First sentence)": df_sample["First sentence"],
    "Gold Reference (second sentence)": df_sample["second sentence"],
    "Expert Score": df_sample["44_experts"],
})

for model_name in MODEL_CONFIG:
    results_df[f"{model_name}_Output"] = results[model_name]
    results_df[f"{model_name}_BLEU"] = bleu_scores[model_name]

# Show selected examples
display_cols = ["Input (First sentence)", "Gold Reference (second sentence)"]
for m in MODEL_CONFIG:
    display_cols += [f"{m}_Output", f"{m}_BLEU"]

results_df[display_cols].head(10)

## 6. Results Presentation & Visualization

In [ ]:
# ── Visualization: BLEU Score Comparison ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

# 1) Bar chart — Mean BLEU
means = {m: np.mean(bleu_scores[m]) for m in MODEL_CONFIG}
bars = axes[0].bar(means.keys(), means.values(), color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title("Mean BLEU Score by Model", fontsize=14, fontweight='bold')
axes[0].set_ylabel("BLEU Score")
axes[0].set_ylim(0, max(means.values()) * 1.35)
for bar, (m, v) in zip(bars, means.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# 2) Box plot — Score distribution
data_for_box = [bleu_scores[m] for m in MODEL_CONFIG]
bp = axes[1].boxplot(data_for_box, labels=list(MODEL_CONFIG.keys()), patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("BLEU Score Distribution by Model", fontsize=14, fontweight='bold')
axes[1].set_ylabel("BLEU Score")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("bleu_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: bleu_comparison.png")

## 7. Error Analysis
Required components:
- **3 worst outputs per model** (lowest BLEU scores)
- **3 Arabic-specific challenges** (dialect confusion, morphology, ambiguity)
- **2 difficult/borderline cases** (high cross-model disagreement)

In [ ]:
# ── Error Analysis: Worst Outputs per Model ──
print("=" * 90)
print("  WORST 3 OUTPUTS PER MODEL")
print("=" * 90)

for model_name in MODEL_CONFIG:
    scores = np.array(bleu_scores[model_name])
    worst_indices = np.argsort(scores)[:3]

    print(f"\n{'─'*70}")
    print(f"  Model: {model_name}")
    print(f"{'─'*70}")
    for rank, idx in enumerate(worst_indices, 1):
        print(f"\n  [{rank}] BLEU = {scores[idx]:.4f}")
        print(f"  Input:     {df_sample.iloc[idx]['First sentence']}")
        print(f"  Gold:      {df_sample.iloc[idx]['second sentence']}")
        print(f"  Model Out: {results[model_name][idx]}")

# ── Arabic-Specific Challenges ──
print(f"\n\n{'='*90}")
print("  ARABIC-SPECIFIC CHALLENGES")
print("=" * 90)
print("""
1. Dialect Variation (تنوّع اللهجات):
   Models trained primarily on MSA may fail to paraphrase dialectal inputs correctly,
   or may convert dialectal expressions into MSA equivalents, losing the original
   register and style. This causes BLEU mismatches even when meaning is preserved.

2. Morphological Richness (الثراء الصرفي):
   Arabic's rich morphology means the same meaning can be expressed with different
   verb forms (أوزان), noun patterns, and pronominal affixes. A valid paraphrase
   may share very few surface tokens with the reference, leading to low BLEU scores
   despite semantic equivalence.

3. Ambiguity & Polysemy (الغموض وتعدد المعاني):
   Arabic words often carry multiple meanings depending on context. For example,
   'عين' can mean eye, spring, or spy. Models may misinterpret the intended sense,
   producing semantically shifted paraphrases.
""")

# ── Difficult / Borderline Cases ──
print(f"{'='*90}")
print("  DIFFICULT / BORDERLINE CASES")
print("=" * 90)

# Find cases where models disagreed most (high variance across models)
per_sample_scores = np.array([bleu_scores[m] for m in MODEL_CONFIG])  # (5, 100)
variance_per_sample = np.var(per_sample_scores, axis=0)
borderline_indices = np.argsort(variance_per_sample)[-2:]  # top 2 highest variance

for rank, idx in enumerate(borderline_indices, 1):
    print(f"\n  Borderline Case {rank} (cross-model BLEU variance = {variance_per_sample[idx]:.4f}):")
    print(f"  Input: {df_sample.iloc[idx]['First sentence']}")
    print(f"  Gold:  {df_sample.iloc[idx]['second sentence']}")
    for m in MODEL_CONFIG:
        print(f"    {m:<12}: BLEU={bleu_scores[m][idx]:.4f} | {results[m][idx]}")

## 8. Human Evaluation & Inter-Annotator Agreement (IAA)
**Annotation Protocol:**
- 20 stratified samples × 5 models = 100 evaluation entries
- Each sample rated by **2 annotators** on a 1–5 scale for:
  - **Meaning preservation** (does the paraphrase keep the original meaning?)
  - **Fluency** (is the output grammatically correct and natural?)
- Disagreements resolved by a **3rd annotator** (majority vote)
- **Inter-Annotator Agreement** reported using Cohen's Kappa

In [ ]:
# ── Human Evaluation Setup ──
# Select a stratified subset for human evaluation (e.g., 20 samples)
human_eval_indices = np.linspace(0, len(df_sample)-1, 20, dtype=int)

human_eval_df = pd.DataFrame()
rows = []
for idx in human_eval_indices:
    for model_name in MODEL_CONFIG:
        rows.append({
            "sample_id": idx,
            "input": df_sample.iloc[idx]["First sentence"],
            "gold_reference": df_sample.iloc[idx]["second sentence"],
            "model": model_name,
            "model_output": results[model_name][idx],
            "bleu": bleu_scores[model_name][idx],
            "annotator_1_meaning": "",   # 1-5 scale
            "annotator_1_fluency": "",   # 1-5 scale
            "annotator_2_meaning": "",   # 1-5 scale
            "annotator_2_fluency": "",   # 1-5 scale
            "annotator_3_meaning": "",   # (tie-breaker, if needed)
            "annotator_3_fluency": "",
            "final_meaning": "",
            "final_fluency": "",
        })

human_eval_df = pd.DataFrame(rows)
human_eval_df.to_csv("human_evaluation_template.csv", index=False, encoding='utf-8-sig')
print(f"Human evaluation template saved: human_evaluation_template.csv")
print(f"  {len(human_eval_df)} entries ({len(human_eval_indices)} samples × {len(MODEL_CONFIG)} models)")
print(f"\nInstructions: Fill in annotator_1/2 columns (1-5 scale), resolve ties with annotator_3.")
human_eval_df.head()

In [ ]:
# ── Compute IAA (run after filling in human_evaluation_template.csv) ──
from sklearn.metrics import cohen_kappa_score

def compute_iaa(csv_path="human_evaluation_template.csv"):
    """Load annotated CSV and compute Inter-Annotator Agreement."""
    filled_df = pd.read_csv(csv_path)

    # Drop rows where annotations are missing
    filled_df = filled_df.dropna(subset=["annotator_1_meaning", "annotator_2_meaning"])

    if len(filled_df) == 0:
        print("No annotations found yet. Please fill in human_evaluation_template.csv first.")
        return None

    kappa_meaning = cohen_kappa_score(
        filled_df["annotator_1_meaning"].astype(int),
        filled_df["annotator_2_meaning"].astype(int),
    )
    kappa_fluency = cohen_kappa_score(
        filled_df["annotator_1_fluency"].astype(int),
        filled_df["annotator_2_fluency"].astype(int),
    )
    print(f"Inter-Annotator Agreement (Cohen's Kappa):")
    print(f"  Meaning preservation: κ = {kappa_meaning:.3f}")
    print(f"  Fluency:              κ = {kappa_fluency:.3f}")

    # Interpret kappa values
    def interpret_kappa(k):
        if k < 0.20: return "Poor"
        elif k < 0.40: return "Fair"
        elif k < 0.60: return "Moderate"
        elif k < 0.80: return "Substantial"
        else: return "Almost Perfect"

    print(f"\n  Meaning agreement level: {interpret_kappa(kappa_meaning)}")
    print(f"  Fluency agreement level: {interpret_kappa(kappa_fluency)}")

    # Aggregate human scores per model
    if "final_meaning" in filled_df.columns:
        valid = filled_df.dropna(subset=["final_meaning", "final_fluency"])
        if len(valid) > 0:
            human_summary = valid.groupby("model").agg(
                mean_meaning=("final_meaning", lambda x: x.astype(float).mean()),
                mean_fluency=("final_fluency", lambda x: x.astype(float).mean()),
                mean_bleu=("bleu", "mean"),
            ).round(3)
            print("\nHuman Evaluation Summary per Model:")
            print(human_summary.to_string())
            return human_summary

    return kappa_meaning, kappa_fluency

# Uncomment the line below after filling in annotations:
# compute_iaa()
print("IAA computation function defined. Run compute_iaa() after completing annotations.")

## 9. Save Full Logs & Results
All inputs, prompts, and outputs are saved for submission as required by the assignment.

In [ ]:
# ── Save All Results ──
# 1) CSV with all outputs and BLEU scores
results_df.to_csv("paraphrasing_results.csv", index=False, encoding='utf-8-sig')

# 2) JSON full log (inputs, prompts, outputs for all models)
logs = {
    "task": "Arabic Paraphrasing (إعادة الصياغة)",
    "task_description": "Rewrite Arabic sentences preserving meaning using different words/structures",
    "metric": "BLEU + Human Evaluation",
    "num_samples": len(df_sample),
    "models": list(MODEL_CONFIG.keys()),
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt_template": USER_PROMPT_TEMPLATE,
    "mean_bleu": {m: float(np.mean(bleu_scores[m])) for m in MODEL_CONFIG},
    "outputs": [],
}

for idx, row in df_sample.iterrows():
    entry = {
        "id": int(idx),
        "input": row["First sentence"],
        "gold_reference": row["second sentence"],
        "expert_score": float(row["44_experts"]),
    }
    for model_name in MODEL_CONFIG:
        entry[f"{model_name}_output"] = results[model_name][idx]
        entry[f"{model_name}_bleu"] = float(bleu_scores[model_name][idx])
    logs["outputs"].append(entry)

with open("paraphrasing_full_log.json", "w", encoding="utf-8") as f:
    json.dump(logs, f, ensure_ascii=False, indent=2)

print("Files saved:")
print("  paraphrasing_results.csv        — tabular results with BLEU scores")
print("  paraphrasing_full_log.json      — complete log (inputs, prompts, outputs)")
print("  human_evaluation_template.csv   — template for human annotation")
print("  bleu_comparison.png             — visualization")

## 10. Limitations

In [ ]:
# ── Limitations Analysis ──
limitations = """
=== LIMITATIONS OF THIS BENCHMARKING STUDY ===

1. BLEU Score Limitations:
   - BLEU relies on n-gram overlap and may not capture semantic equivalence.
   - A valid Arabic paraphrase can use entirely different vocabulary, leading to
     low BLEU despite perfect meaning preservation.
   - Example: "ذهب إلى المدرسة" and "توجّه نحو المؤسسة التعليمية" have the same
     meaning but minimal word overlap.

2. Dataset Size & Balance:
   - Only 100 samples were used due to API rate limits and GPU constraints.
   - The dialectal Arabic coverage may be limited since the source dataset is
     primarily MSA-oriented.
   - Classical Arabic examples are underrepresented.

3. Model Access Constraints:
   - API models (ChatGPT, Gemini, Fanar) were accessed via free-tier APIs with rate
     limits, which may affect availability and response quality.
   - HF models (ALLaM, Jais) were run in float16 on a single GPU, which
     may reduce output quality compared to full-precision inference.

4. Evaluation Protocol:
   - temperature=0 (greedy decoding) was used for reproducibility, but this may
     not reflect typical creative paraphrasing use cases.
   - The same prompt was used for all models, but some models may perform better
     with model-specific prompt engineering.

5. Human Evaluation Scale:
   - Only 20 samples (out of 100) were selected for human evaluation due to
     time and annotator availability constraints.
   - The 1-5 Likert scale is subjective and may vary across annotators.

6. Arabic-Specific Limitations:
   - No diacritization normalization was applied, which may affect BLEU scores.
   - Dialectal inputs may be misinterpreted by models trained primarily on MSA.
"""
print(limitations)

## 11. Conclusions, Comments & Recommendations

In [ ]:
# ── Final Conclusions & Recommendations ──

# Generate a summary report
print("=" * 90)
print("  FINAL CONCLUSIONS & RECOMMENDATIONS")
print("=" * 90)

# Rank models by mean BLEU
ranked = sorted(
    [(m, np.mean(bleu_scores[m])) for m in MODEL_CONFIG],
    key=lambda x: x[1],
    reverse=True,
)

print("\n1. MODEL RANKING (by Mean BLEU):")
for rank, (model, score) in enumerate(ranked, 1):
    n_err = sum(1 for o in results[model] if o.startswith("ERROR"))
    print(f"   {rank}. {model:<12} — Mean BLEU: {score:.4f}  (Errors: {n_err})")

best_model = ranked[0][0]
worst_model = ranked[-1][0]

print(f"\n2. KEY FINDINGS:")
print(f"   • Best performing model: {best_model} (BLEU = {ranked[0][1]:.4f})")
print(f"   • Lowest performing model: {worst_model} (BLEU = {ranked[-1][1]:.4f})")
print(f"   • BLEU score range across models: {ranked[-1][1]:.4f} – {ranked[0][1]:.4f}")

# Check error rates
print(f"\n3. RELIABILITY (Error Rates):")
for model in MODEL_CONFIG:
    n_err = sum(1 for o in results[model] if o.startswith("ERROR"))
    pct = n_err / len(results[model]) * 100
    status = "✓ Reliable" if pct < 5 else "⚠ Issues" if pct < 20 else "✗ Unreliable"
    print(f"   {model:<12}: {n_err}/{len(results[model])} errors ({pct:.1f}%) — {status}")

print(f"""
4. ARABIC-SPECIFIC OBSERVATIONS:
   • Models trained on large multilingual corpora (ChatGPT, Gemini) tend to
     produce more diverse paraphrases but may sometimes shift the register.
   • Arabic-focused models (ALLaM, Jais, Fanar) may preserve the Arabic
     style better but sometimes produce less varied paraphrases.
   • Larger models (e.g., Fanar 27B) tend to produce more fluent paraphrases but
     can be less literal than smaller models.
   • All models struggle with dialectal inputs and Classical Arabic texts.

5. RECOMMENDATIONS:
   • For production Arabic paraphrasing: use the top-ranked model ({best_model})
     but always include human review for critical applications.
   • Combine BLEU with semantic similarity metrics (e.g., BERTScore with
     Arabic BERT) for more accurate automatic evaluation.
   • Fine-tuning Arabic-specific models on paraphrasing datasets could
     significantly improve performance.
   • Future work should include larger and more dialectally diverse datasets
     covering Gulf, Egyptian, Levantine, and Maghrebi dialects.
   • Consider using SARI metric alongside BLEU for a more comprehensive
     evaluation of paraphrasing quality.
""")

# ── Final Summary Table ──
print("=" * 90)
print("  SUMMARY TABLE")
print("=" * 90)
summary_data = []
for model in MODEL_CONFIG:
    s = bleu_scores[model]
    n_err = sum(1 for o in results[model] if o.startswith("ERROR"))
    summary_data.append({
        "Model": model,
        "Backend": MODEL_CONFIG[model]["backend"],
        "Mean BLEU": round(np.mean(s), 4),
        "Median BLEU": round(np.median(s), 4),
        "Std BLEU": round(np.std(s), 4),
        "Min BLEU": round(np.min(s), 4),
        "Max BLEU": round(np.max(s), 4),
        "Errors": n_err,
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv("model_summary.csv", index=False)
print("\nSaved: model_summary.csv")

In [ ]:
# ── Additional Visualization: Heatmap of per-sample BLEU across models ──
fig, ax = plt.subplots(figsize=(14, 6))
heatmap_data = pd.DataFrame({m: bleu_scores[m] for m in MODEL_CONFIG})
sns.heatmap(
    heatmap_data.T,
    cmap='YlOrRd',
    ax=ax,
    xticklabels=False,
    yticklabels=list(MODEL_CONFIG.keys()),
    cbar_kws={'label': 'BLEU Score'},
)
ax.set_xlabel("Sample Index (100 sentences)")
ax.set_ylabel("Model")
ax.set_title("Per-Sample BLEU Score Heatmap Across All 5 Models", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("bleu_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: bleu_heatmap.png")